In [ ]:
# input
tax_nodes = "../../../../_database/taxonomy/nodes.dmp"
tax_names = "../../../../_database/taxonomy/names.dmp"

# param
target_ids = [
    590, # Salmonella: 590 (genus)
    287, # Pseudomonas aeruginosa
    1280, # Staphylococcus aureus
    562, # Escherichia coli
    573 # Klebsiella pneumoniae
]

# output
tax_ids_file = "./data/tax_ids.tsv"

In [2]:
import pandas as pd

# 读取 nodes.dmp
nodes = pd.read_csv(
    tax_nodes, sep="\t\|\t", header=None, engine="python",
    names=["tax_id", "parent_tax_id", "rank", "embl_code", "division_id",
           "inherited_div_flag", "genetic_code_id", "inherited_gc_flag",
           "mitochondrial_genetic_code_id", "inherited_mgc_flag",
           "genbank_hidden_flag", "hidden_subtree_root_flag", "comments"],
    usecols=["tax_id", "parent_tax_id", "rank"]
)

# 读取 names.dmp
names = pd.read_csv(
    tax_names, sep="\t\|\t", header=None, engine="python",
    names=["tax_id", "name_txt", "unique_name", "name_class"]
)
names = names[names["name_class"] == "scientific name\t|"].drop("name_class", axis=1)

# 合并 nodes 和 names，得到 TaxID 和分类信息
taxonomy = pd.merge(nodes, names, on="tax_id")
len(taxonomy)

2628095

In [3]:
class TaxNode:
    def __init__(self, id: int, parent_id: int | None = None, rank: str | None = None, name: str | None = None) -> None:
        self.id = id
        self.parent_id = parent_id
        self.rank = rank
        self.name = name
        self.children = []

    @property
    def is_root(self):
        return self.parent_id == self.id

    @property
    def is_leaf(self):
        return len(self.children) == 0
    
    def update_info(self, parent_id: int, rank: str, name: str):
        self.parent_id = parent_id
        self.rank = rank
        self.name = name

    def add_child(self, node: 'TaxNode'):
        self.children.append(node)

    def __repr__(self) -> str:
        return f"'{self.id}, {self.parent_id}, {self.rank}, {self.name}'"

In [4]:
import tqdm

id2node = dict()

for _, row in tqdm.tqdm(taxonomy.iterrows(), total = len(taxonomy)):
    id = row['tax_id']
    parent_id = row['parent_tax_id']
    rank = row['rank']
    name = row['name_txt']

    if id not in id2node:
        node = TaxNode(id, parent_id, rank, name)
        id2node[id] = node
    else:
        id2node[id].update_info(parent_id, rank, name)

    if parent_id not in id2node:
        node = TaxNode(parent_id)
        node.add_child(id2node[id])
        id2node[parent_id] = node
    else:
        if not id2node[id].is_root:
            id2node[parent_id].add_child(id2node[id])

100%|██████████| 2628095/2628095 [01:54<00:00, 22865.53it/s]


In [5]:
def get_all_children(node: TaxNode):
    if node.is_leaf: 
        return {node.id}

    children_ids = set()
    for c in node.children:
        children_ids |= get_all_children(c)
    return children_ids

In [6]:
targets = []
for c in target_ids:
    targets.append(c)
    targets.extend(get_all_children(id2node[c]))
len(targets)

14900

In [9]:
with open(tax_ids_file, "w") as f:
    f.writelines("\n".join([str(t) for t in targets]))